## 5: AgentCore Runtime

**What you'll learn:** Deploy agents to production with automatic scaling, managed infrastructure, and enterprise reliability using just 4 lines of code.

**Why it matters:** Runtime eliminates infrastructure management. Your agent runs in a serverless environment with auto-scaling, health monitoring, and built-in observability.

**Real-world value:** Focus on agent logic, not DevOps. Runtime handles container orchestration, scaling, load balancing, and high availability automatically.

**Analogy:** Like deploying to a cloud platform (Heroku, Vercel): you push code, the platform handles servers, scaling, and monitoring. You focus on features, not infrastructure.

![Runtime](images/Runtime.png)

---

**Prerequisites:** Completed Identity, Gateway & Memory.

### Import Required Libraries

In [14]:
import os

# Set the AWS profile to match your shell environment
os.environ['AWS_PROFILE'] = 'workshop-profile'

import json
import time
import boto3
from bedrock_agentcore_starter_toolkit import Runtime

# Get AWS session information
session = boto3.Session()
region = session.region_name or 'us-west-2'

sts = session.client('sts')
identity = sts.get_caller_identity()
account_id = identity['Account']

print(f"✅ Account ID: {account_id}")
print(f"✅ Region: {region}")

✅ Account ID: 625579972148
✅ Region: us-west-2


### Load Configurations

In [15]:
# Load Cognito configuration from 2
with open('cognito_config.json', 'r') as f:
    cognito_config = json.load(f)

print("✅ Loaded Cognito configuration")
print(f"Client ID: {cognito_config.get('client_id')}")
print(f"Discovery URL: {cognito_config.get('discovery_url')}")

# Load memory and knowledge base configurations from 1
with open('memory_config.json', 'r') as f:
    memory_config = json.load(f)
memory_id = memory_config['memory_id']

with open('kb_config.json', 'r') as f:
    kb_config = json.load(f)
kb_id = kb_config['kb_id']

print(f"\nMemory ID: {memory_id}")
print(f"Knowledge Base ID: {kb_id}")

✅ Loaded Cognito configuration
Client ID: 6dacsfqdd4b9imggc4dnlsiebv
Discovery URL: https://cognito-idp.us-west-2.amazonaws.com/us-west-2_QuW2OOBnZ/.well-known/openid-configuration

Memory ID: ReturnRefundAssisantMemory-3ar3Bl9YAh
Knowledge Base ID: EK2IHAXS8Q


### Step 1: Preparing Your Agent for AgentCore Runtime

To make your agent runtime-ready, add just 4 lines of code:
1. Import `BedrockAgentCoreApp`
2. Initialize the app
3. Decorate your function with `@app.entrypoint`
4. Call `app.run()`

The agent is configured with:
- **Knowledge Base**: Policy questions via `retrieve` tool
- **Gateway Tools**: Refund management (create, list, approve)
- **Three Memory Strategies**:
  - SEMANTIC: Retrieves factual details from past conversations
  - USER_PREFERENCE: Recalls customer preferences and behavior patterns
  - SUMMARY: Provides conversation context and summaries

In [16]:
%%writefile ./agent_runtime.py
import os
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import retrieve, current_time
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
from utils.agent_memory import REGION, SESSION_ID, ACTOR_ID
from utils.identity_ssm_utils import get_cognito_token_with_scope

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
bedrock_model = BedrockModel(model_id=MODEL_ID, temperature=0.3)
app = BedrockAgentCoreApp()

# Get configuration from environment variables
kb_id = os.environ.get("KNOWLEDGE_BASE_ID", "NOT AVAILABLE")
memory_id = os.environ.get("MEMORY_ID")
gateway_url = os.environ.get("GATEWAY_URL")
cognito_client_id = os.environ.get("COGNITO_CLIENT_ID")
cognito_client_secret = os.environ.get("COGNITO_CLIENT_SECRET")
cognito_discovery_url = os.environ.get("COGNITO_DISCOVERY_URL")

if not memory_id:
    raise Exception("Environment variable MEMORY_ID is required")

def create_mcp_client():
    """Create MCP client for gateway access"""
    token = get_cognito_token_with_scope(
        cognito_client_id,
        cognito_client_secret,
        cognito_discovery_url,
        "workshop-api/read workshop-api/write"
    )
    return MCPClient(
        lambda: streamablehttp_client(
            gateway_url,
            headers={"Authorization": f"Bearer {token}"},
        )
    )

system_prompt = f"""You are an Amazon Returns & Refunds assistant with access to:
- Knowledge Base (retrieve tool with knowledgeBaseId="{kb_id}") for policy questions
- Gateway tools for refund management (create, list, approve refund requests)
- Customer conversation history and preferences through memory

Use conversation history to provide personalized assistance. Reference past interactions when relevant.
For refund operations, use user_id 'user456' as default if not specified."""

@app.entrypoint
def invoke(payload, context=None):
    """AgentCore Runtime entrypoint"""
    session_id = context.session_id if context else SESSION_ID
    actor_id = payload.get("actor_id", ACTOR_ID)
    
    # Configure memory with retrieval settings
    agentcore_memory_config = AgentCoreMemoryConfig(
        memory_id=memory_id,
        session_id=session_id,
        actor_id=actor_id,
        retrieval_config={
            f"returns/customer/{actor_id}/semantic": RetrievalConfig(top_k=3, relevance_score=0.2),
            f"returns/customer/{actor_id}/preferences": RetrievalConfig(top_k=3, relevance_score=0.2),
            f"returns/customer/{actor_id}/{session_id}/summary": RetrievalConfig(top_k=2, relevance_score=0.2)
        }
    )
    
    session_manager = AgentCoreMemorySessionManager(
        agentcore_memory_config=agentcore_memory_config,
        region_name=REGION
    )
    
    # Create MCP client and use it within context manager
    mcp_client = create_mcp_client()
    
    with mcp_client:
        # Get gateway tools from MCP client while it's active
        gateway_tools = list(mcp_client.list_tools_sync())
        
        # Create agent with all tools
        agent = Agent(
            model=bedrock_model,
            tools=[retrieve, current_time] + gateway_tools,
            system_prompt=system_prompt,
            session_manager=session_manager
        )
        
        user_input = payload.get("prompt", "")
        response = agent(user_input)
        return response.message["content"][0]["text"]

if __name__ == "__main__":
    app.run()

Overwriting ./agent_runtime.py


### What Happens Behind the Scenes?

`BedrockAgentCoreApp` automatically creates an HTTP server on port 8080, implements `/invocations` and `/ping` endpoints, and handles proper content types and error handling.

### Step 2: Configure the Runtime Deployment

**⚠️ IMPORTANT: If you get ResourceNotFoundException, run the cell below first!**

Configure deployment settings including entrypoint, execution role, and Cognito authentication.

In [17]:
import os

# Remove old configuration if it exists
config_file = '.bedrock_agentcore.yaml'
if os.path.exists(config_file):
    print("🧹 Found existing configuration file...")
    os.remove(config_file)
    print("✅ Removed old configuration. Will create fresh deployment.")
else:
    print("✅ No old configuration found. Ready for fresh deployment.")

🧹 Found existing configuration file...
✅ Removed old configuration. Will create fresh deployment.


In [18]:
# Initialize the AgentCore runtime toolkit
agentcore_runtime = Runtime()

# Configure the AgentCore agent deployment
response = agentcore_runtime.configure(
    entrypoint="agent_runtime.py",
    auto_create_ecr=True,
    execution_role=cognito_config.get("execution_role"),
    auto_create_execution_role=False,
    memory_mode="NO_MEMORY",
    requirements_file="requirements.txt",
    region=region,
    agent_name="returns_refunds_agent",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
)

print("Configuration completed:", response)

Entrypoint parsed: file=C:\dev\sample-aiml-solution-labs\workshop\build-and-test-ai-agents-with-kiro-deploy-with-amazon-bedrock-agentcore\agentcore-lab\agent_runtime.py, bedrock_agentcore_name=agent_runtime
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: returns_refunds_agent


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

📄 Using existing Dockerfile: 
c:\dev\sample-aiml-solution-labs\workshop\build-and-test-ai-agents-with-kiro-deploy-with-amazon-bedrock-agentcore\a
gentcore-lab\Dockerfile

Generated .dockerignore: c:\dev\sample-aiml-solution-labs\workshop\build-and-test-ai-agents-with-kiro-deploy-with-amazon-bedrock-agentcore\agentcore-lab\.dockerignore
Setting 'returns_refunds_agent' as default agent
Bedrock AgentCore configured: c:\dev\sample-aiml-solution-labs\workshop\build-and-test-ai-agents-with-kiro-deploy-with-amazon-bedrock-agentcore\agentcore-lab\.bedrock_agentcore.yaml


Configuration completed: config_path=WindowsPath('c:/dev/sample-aiml-solution-labs/workshop/build-and-test-ai-agents-with-kiro-deploy-with-amazon-bedrock-agentcore/agentcore-lab/.bedrock_agentcore.yaml') dockerfile_path=WindowsPath('c:/dev/sample-aiml-solution-labs/workshop/build-and-test-ai-agents-with-kiro-deploy-with-amazon-bedrock-agentcore/agentcore-lab/Dockerfile') dockerignore_path=WindowsPath('c:/dev/sample-aiml-solution-labs/workshop/build-and-test-ai-agents-with-kiro-deploy-with-amazon-bedrock-agentcore/agentcore-lab/.dockerignore') runtime='None' runtime_type=None region='us-west-2' account_id='625579972148' execution_role='arn:aws:iam::625579972148:role/ReturnsRefundsAssistantBedrockAgentCoreRole-us-west-2' ecr_repository=None auto_create_ecr=True s3_path=None auto_create_s3=False memory_id=None network_mode='PUBLIC' network_subnets=None network_security_groups=None network_vpc_id=None


### View Generated Configuration

In [19]:
! type .bedrock_agentcore.yaml

default_agent: returns_refunds_agent
agents:
  returns_refunds_agent:
    name: returns_refunds_agent
    language: python
    node_version: null
    entrypoint: C:/dev/sample-aiml-solution-labs/workshop/build-and-test-ai-agents-with-kiro-deploy-with-amazon-bedrock-agentcore/agentcore-lab/agent_runtime.py
    deployment_type: container
    runtime_type: null
    platform: linux/arm64
    container_runtime: none
    source_path: null
    aws:
      execution_role: arn:aws:iam::625579972148:role/ReturnsRefundsAssistantBedrockAgentCoreRole-us-west-2
      execution_role_auto_create: false
      account: '625579972148'
      region: us-west-2
      ecr_repository: null
      ecr_auto_create: true
      s3_path: null
      s3_auto_create: false
      network_configuration:
        network_mode: PUBLIC
        network_mode_config: null
      protocol_configuration:
        server_protocol: HTTP
      observability:
        enabled: true
      lifecycle_configuration:
        idle_runtime_ses

### Step 3: Launch the Agent

Deploy to AgentCore Runtime. This creates a CodeBuild pipeline, ECR repository, and runtime components.

**Note:** If you previously deleted the agent or are re-running this lab, the launch will create a new agent instance.

In [20]:
# Load gateway configuration
with open('gateway_config.json', 'r') as f:
    gateway_config = json.load(f)

print("🚀 Launching agent to AgentCore Runtime...")
print("   This will:")
print("   1. Build a Docker container with your agent code")
print("   2. Push the container to Amazon ECR")
print("   3. Deploy to AgentCore Runtime")
print("   4. Configure authentication and environment variables")
print("\n⏳ This process typically takes 5-10 minutes...\n")

try:
    launch_result = agentcore_runtime.launch(
        env_vars={
            "MEMORY_ID": memory_id,
            "KNOWLEDGE_BASE_ID": kb_id,
            "GATEWAY_URL": gateway_config['gateway_url'],
            "COGNITO_CLIENT_ID": cognito_config['client_id'],
            "COGNITO_CLIENT_SECRET": cognito_config['client_secret'],
            "COGNITO_DISCOVERY_URL": cognito_config['discovery_url']
        },
        auto_update_on_conflict=True
    )
    
    print(f"✅ Launch initiated successfully!")
    print(f"   Agent ARN: {launch_result.agent_arn}")
    
    # Save runtime configuration to JSON file
    runtime_config = {
        "agent_arn": launch_result.agent_arn
    }
    with open('runtime_config.json', 'w') as f:
        json.dump(runtime_config, f, indent=2)
    
    print("\n✅ Runtime configuration saved to runtime_config.json")
    print("\n💡 Proceed to the next cell to monitor deployment status.")
    
except Exception as e:
    print(f"❌ Launch failed: {e}")
    print("\nCommon issues:")
    print("1. Docker/Finch/Podman not running - start your container runtime")
    print("2. Insufficient IAM permissions - check your AWS credentials")
    print("3. ECR repository issues - verify ECR access")
    print("4. Configuration file missing - ensure gateway_config.json exists")
    raise

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'returns_refunds_agent' to account 625579972148 (us-west-2)
Generated image tag: 20260123-121324-299
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: returns_refunds_agent


🚀 Launching agent to AgentCore Runtime...
   This will:
   1. Build a Docker container with your agent code
   2. Push the container to Amazon ECR
   3. Deploy to AgentCore Runtime
   4. Configure authentication and environment variables

⏳ This process typically takes 5-10 minutes...



ECR repository available: 625579972148.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-returns_refunds_agent
Using execution role from config: arn:aws:iam::625579972148:role/ReturnsRefundsAssistantBedrockAgentCoreRole-us-west-2
Preparing CodeBuild project and uploading source...


✅ Reusing existing ECR repository: 625579972148.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-returns_refunds_agent


Getting or creating CodeBuild execution role for agent: returns_refunds_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-8eda8502c8
Reusing existing CodeBuild execution role: arn:aws:iam::625579972148:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-8eda8502c8
Using dockerignore.template with 46 patterns for zip filtering
Uploaded source to S3: returns_refunds_agent/source.zip
Updated CodeBuild project: bedrock-agentcore-returns_refunds_agent-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.9s
🔄 DOWNLOAD_SOURCE started (total: 9s)
✅ DOWNLOAD_SOURCE completed in 1.1s
🔄 PRE_BUILD started (total: 10s)
✅ PRE_BUILD completed in 1.1s
🔄 BUILD started (total: 11s)
✅ BUILD completed in 18.2s
🔄 POST_BUILD started (total: 30s)
✅ POST_BUILD completed in 14.8s
🔄 FINALIZING started (total: 44s)
✅ FINALIZING complete

✅ Launch initiated successfully!
   Agent ARN: arn:aws:bedrock-agentcore:us-west-2:625579972148:runtime/returns_refunds_agent-HSGBtQFj7x

✅ Runtime configuration saved to runtime_config.json

💡 Proceed to the next cell to monitor deployment status.


### Step 4: Check Deployment Status

Monitor the deployment progress. The agent goes through several stages: building the container image, pushing to ECR, and deploying to AgentCore Runtime.

In [21]:
# Wait for the agent to be ready
print("🔍 Checking agent deployment status...\n")

try:
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    
    end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
    
    while status not in end_status:
        print(f"⏳ Current status: {status}")
        time.sleep(10)
        status_response = agentcore_runtime.status()
        status = status_response.endpoint["status"]
    
    if status == "READY":
        print(f"\n✅ Agent is {status} and ready to use!")
    else:
        print(f"\n⚠️ Agent deployment ended with status: {status}")
        print("Check CloudWatch logs for details.")
        
except Exception as e:
    print(f"⚠️ Error checking status: {e}")
    print("\nThis might happen if:")
    print("1. The agent is still being created (wait a few minutes)")
    print("2. The agent was deleted and needs to be re-launched")
    print("3. There's a configuration mismatch")
    print("\n💡 If the agent was previously deleted, re-run the launch cell above.")

🔍 Checking agent deployment status...



Retrieved Bedrock AgentCore status for: returns_refunds_agent



✅ Agent is READY and ready to use!


### Step 5: Test Your Deployed Agent

Test the agent with Knowledge Base queries, Gateway tool calls, and memory-aware responses.

**Note:** If you get a 424 error, the agent may still be initializing. Wait a minute and try again.

### Test 1: Knowledge Base Integration

Test the agent's ability to query the knowledge base for return policy information.

In [22]:
from utils.identity_ssm_utils import reauthenticate_user

# Get fresh bearer token
bearer_token = reauthenticate_user(
    cognito_config.get("client_id"),
    cognito_config.get("client_secret")
)

print("\n" + "="*80)
print("TEST 1: Knowledge Base Query")
print("="*80)

# Test 1: Knowledge Base query
query1 = "What's the return policy for electronics in the US?"
print(f"\nQuery: {query1}\n")

try:
    response1 = agentcore_runtime.invoke(
        {"prompt": query1},
        bearer_token=bearer_token
    )


    # Format the response for better readability
    from IPython.display import display, Markdown
    
    print("Response:")
    print("-" * 80)
    
    if isinstance(response1, dict):
        # If response is a dict, extract the actual text content
        response_text = response1.get('response', str(response1))
        
        # Remove extra quotes and unescape newlines
        if isinstance(response_text, str):
            # Remove surrounding quotes if present
            response_text = response_text.strip('"\'')
            # Replace escaped newlines with actual newlines
            response_text = response_text.replace('\\n', '\n')
            # Replace escaped quotes
            response_text = response_text.replace('\\"', '"')
            response_text = response_text.replace("\\'", "'")
            
        # Display as rendered markdown
        display(Markdown(response_text))
    else:
        print(response1)
    
    print("-" * 80 + "\n")


except Exception as e:
    print(f"❌ Error: {e}")
    print("\nTroubleshooting tips:")
    print("1. Check if agent status is READY (run the status cell above)")
    print("2. Wait 1-2 minutes for the agent to fully initialize")
    print("3. Verify the gateway_config.json and cognito_config.json files exist")
    print("4. Check CloudWatch logs for the agent runtime")

print("="*80)

Using JWT authentication



TEST 1: Knowledge Base Query

Query: What's the return policy for electronics in the US?

Response:
--------------------------------------------------------------------------------


Based on the Amazon return policy for the US, here's what you need to know about returning electronics:

## **US Electronics Return Policy**

**General Return Window:**
- Most electronics can be returned within a standard return window (typically 30 days for most items)
- Returns are generally **free** at over 8,000 convenient locations, usually within a 5-mile radius of your address
- Most returns don't need to be boxed or labeled

**Condition Requirements:**
Your electronics must be returned in:
- **Original or unused condition** with tags attached
- **Original manufacturer's packaging** including all components, accessories, manuals, and inserts
- Hygiene seals and liners intact (if applicable)

**Important for Electronics:**
- If you've stored personal information on the device (computers, electronics, etc.), **you must completely erase this information** before returning it following the manufacturer's instructions
- Amazon will not be liable for any misuse of personal information left on devices

**Special Cases:**
- **Amazon Digital Devices** (Echo, Fire TV Stick, Kindle E-Readers): 7 days for replacement only
- **Damaged/Defective Items**: If you receive damaged or defective electronics, Amazon will typically provide a full refund or free replacement

**Return Process:**
- Check your "Return Request" tab in your Order History for the specific "return by date"
- Follow the return instructions for your specific item

Since I see from your history that you have a damaged product order (ORD-12345), you should be eligible for a refund or replacement. Would you like help with that?

--------------------------------------------------------------------------------



### Test 2: Gateway Tool - Create Refund Request

Test the agent's ability to create a refund request using the gateway tool.

In [23]:
from utils.identity_ssm_utils import reauthenticate_user

# Get fresh bearer token
bearer_token = reauthenticate_user(
    cognito_config.get("client_id"),
    cognito_config.get("client_secret")
)

print("\n" + "="*80)
print("TEST 2: Create Refund Request (Gateway Tool)")
print("="*80)

# Test 2: Create refund request
query2 = "Create a refund request for order ORD-12345 with amount $49.99 because the product arrived damaged. Use user_id user456"
print(f"\nQuery: {query2}\n")

try:
    response2 = agentcore_runtime.invoke(
        {"prompt": query2},
        bearer_token=bearer_token
    )


    # Format the response for better readability
    from IPython.display import display, Markdown
    
    print("Response:")
    print("-" * 80)
    
    if isinstance(response2, dict):
        # If response is a dict, extract the actual text content
        response_text = response2.get('response', str(response2))
        
        # Remove extra quotes and unescape newlines
        if isinstance(response_text, str):
            # Remove surrounding quotes if present
            response_text = response_text.strip('"\'')
            # Replace escaped newlines with actual newlines
            response_text = response_text.replace('\\n', '\n')
            # Replace escaped quotes
            response_text = response_text.replace('\\"', '"')
            response_text = response_text.replace("\\'", "'")
            
        # Display as rendered markdown
        display(Markdown(response_text))
    else:
        print(response2)
    
    print("-" * 80 + "\n")


except Exception as e:
    print(f"❌ Error: {e}\n")

print("="*80)

Using JWT authentication



TEST 2: Create Refund Request (Gateway Tool)

Query: Create a refund request for order ORD-12345 with amount $49.99 because the product arrived damaged. Use user_id user456

Response:
--------------------------------------------------------------------------------


Perfect! I've successfully created a refund request for you. Here are the details:

**Refund Request Created:**
- **Refund Request ID:** 8a9a7bee-aa81-4d45-88ab-1d6b42818a64
- **Order ID:** ORD-12345
- **Amount:** $49.99
- **Reason:** Product arrived damaged
- **Status:** Pending

Your refund request is now pending review. You can use the refund request ID to track the status of your request. Since the product arrived damaged, you should be eligible for a full refund or replacement according to Amazon's return policy.

Is there anything else you'd like me to help you with regarding this refund or any other returns?

--------------------------------------------------------------------------------



### Test 3: Gateway Tool - List Refund Requests

Test the agent's ability to list refund requests using the gateway tool.

In [24]:
from utils.identity_ssm_utils import reauthenticate_user

# Get fresh bearer token
bearer_token = reauthenticate_user(
    cognito_config.get("client_id"),
    cognito_config.get("client_secret")
)

print("\n" + "="*80)
print("TEST 3: List Refund Requests (Gateway Tool)")
print("="*80)

# Test 3: List refund requests
query3 = "List all refund requests for user_id user456"
print(f"\nQuery: {query3}\n")

try:
    response3 = agentcore_runtime.invoke(
        {"prompt": query3},
        bearer_token=bearer_token
    )


    # Format the response for better readability
    from IPython.display import display, Markdown
    
    print("Response:")
    print("-" * 80)
    
    if isinstance(response3, dict):
        # If response is a dict, extract the actual text content
        response_text = response3.get('response', str(response3))
        
        # Remove extra quotes and unescape newlines
        if isinstance(response_text, str):
            # Remove surrounding quotes if present
            response_text = response_text.strip('"\'')
            # Replace escaped newlines with actual newlines
            response_text = response_text.replace('\\n', '\n')
            # Replace escaped quotes
            response_text = response_text.replace('\\"', '"')
            response_text = response_text.replace("\\'", "'")
            
        # Display as rendered markdown
        display(Markdown(response_text))
    else:
        print(response3)
    
    print("-" * 80 + "\n")


except Exception as e:
    print(f"❌ Error: {e}\n")

print("="*80)

Using JWT authentication



TEST 3: List Refund Requests (Gateway Tool)

Query: List all refund requests for user_id user456

Response:
--------------------------------------------------------------------------------


Here are all the refund requests for user456:

**Refund Requests Summary:**
Total Requests: 2

| Refund Request ID | Order ID | Amount | Reason | Status | Approver Notes |
|---|---|---|---|---|---|
| 22a718a9-176a-4927-882f-132bbabb064b | ORD-12345 | $49.99 | product arrived damaged | Pending | — |
| 8a9a7bee-aa81-4d45-88ab-1d6b42818a64 | ORD-12345 | $49.99 | Product arrived damaged | Pending | — |

Both refund requests are currently **pending** review. They're both for the same order (ORD-12345) with the same amount and reason. You may want to check if one of these is a duplicate that needs to be cancelled.

Would you like me to help you with anything else, such as approving/rejecting a request or creating a new one?

--------------------------------------------------------------------------------



### Summary

You've deployed your agent to production with just 4 lines of code changes. The agent now runs in a scalable, managed environment with automatic scaling and enterprise reliability.

### Next Steps

- **6: AgentCore Observability** - Monitor and trace your agent